In [ ]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

stores = ["Warszawa", "Kraków", "Gdańsk", "Wrocław"]
categories = ["elektronika", "odzież", "żywność", "książki"]

tx_counter = 1

def generate_transaction():
    global tx_counter

    transaction = {
        "tx_id": f"TX{tx_counter:04d}",
        "user_id": f"u{random.randint(1,20):02d}",
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(stores),
        "category": random.choice(categories),
        "timestamp": datetime.now().isoformat()
    }

    tx_counter += 1
    return transaction

# Pętla wysyłająca dane do Kafka topic: transactions
while True:
    transaction = generate_transaction()

    producer.send('transactions', value=transaction)

    print(transaction)

    time.sleep(1)

In [ ]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
from collections import defaultdict
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='anomaly-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)


user_transactions = defaultdict(list)

print("Nasłuchuję anomalii prędkości transakcji...")

for message in consumer:

    transaction = message.value

    user_id = transaction['user_id']
    tx_id = transaction['tx_id']


    tx_time = datetime.fromisoformat(transaction['timestamp'])


    user_transactions[user_id].append(tx_time)


    time_limit = tx_time - timedelta(seconds=60)


    user_transactions[user_id] = [
        t for t in user_transactions[user_id]
        if t >= time_limit
    ]


    if len(user_transactions[user_id]) > 3:

        print(
            f"ALERT ANOMALY: użytkownik {user_id} "
            f"wykonał {len(user_transactions[user_id])} "
            f"transakcje w ciągu 60 sekund! "
            f"(ostatnia: {tx_id})"
        )